# Würfel-Zerfall: Exponentieller Fit

Auswertung des Würfel-Experiments (Start mit 99 bzw. 100 Würfeln, "zerfallen" bei Augenzahl 3 bzw. 6). Benötigt `scipy` (siehe `python_setup.md`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
# Datensatz 1: N0 = 99, "zerfallen" bei Augenzahl 3
würfel_anzahl_3 = [19, 10, 13, 7, 8, 10, 6, 6, 3, 2, 6, 2, 3, 2, 0, 2]

würfel = [99]
for würfel_tot in würfel_anzahl_3:
    würfel.append(würfel[-1] - würfel_tot)

verbleibend_1 = np.array(würfel)
runde_1 = np.arange(len(verbleibend_1))

# Datensatz 2: N0 = 100, "zerfallen" bei Augenzahl 6
würfel_anzahl_6 = [19, 12, 8, 7, 8, 5, 10, 3, 3, 5, 3, 2, 3, 2, 1, 2, 3, 0, 1, 2, 0, 0, 0, 0, 1]

würfel = [100]
for würfel_tot in würfel_anzahl_6:
    würfel.append(würfel[-1] - würfel_tot)

verbleibend_2 = np.array(würfel)
runde_2 = np.arange(len(verbleibend_2))

print(verbleibend_1)
print(verbleibend_2)

In [ ]:
### Alternative Lösung (Claude Code): kumulative Summe statt Schleife

# Datensatz 1: N0 = 99, "zerfallen" bei Augenzahl 3
entfernt_1 = np.array([19, 10, 13, 7, 8, 10, 6, 6, 3, 2, 6, 2, 3, 2, 0, 2])
N0_1 = 99
verbleibend_1_alt = N0_1 - np.concatenate(([0], np.cumsum(entfernt_1)))

# Datensatz 2: N0 = 100, "zerfallen" bei Augenzahl 6
entfernt_2 = np.array([19, 12, 8, 7, 8, 5, 10, 3, 3, 5, 3, 2, 3, 2, 1, 2, 3, 0, 1, 2, 0, 0, 0, 0, 1])
N0_2 = 100
verbleibend_2_alt = N0_2 - np.concatenate(([0], np.cumsum(entfernt_2)))

# beide Wege liefern dasselbe Ergebnis
print(np.array_equal(verbleibend_1, verbleibend_1_alt))
print(np.array_equal(verbleibend_2, verbleibend_2_alt))

## Fit-Modell

$$N(t) = N_0 \cdot e^{-\lambda t}$$

In [ ]:
def exp_zerfall(t, N0, lam):
    return N0 * np.exp(-lam * t)

## `curve_fit`: Argumente und Rückgabe

`curve_fit(f, xdata, ydata, p0=...)`

**Argumente:**
- `f` — die Modellfunktion. Erstes Argument ist die unabhängige Variable (hier `t`), alle weiteren sind die zu fittenden Parameter (hier `N0`, `lam`).
- `xdata`, `ydata` — die Messdaten (hier `runde_1`, `verbleibend_1`).
- `p0` — Startwerte für die Parameter (Ausgangspunkt der Optimierung, hier `[99, 0.2]`).

**Rückgabe:** ein Tupel `(popt, pcov)`
- `popt` — Array der optimalen Parameter, in der Reihenfolge wie in `f` definiert (hier `params_1` = `[N0, lam]`).
- `pcov` — Kovarianzmatrix der Parameter (Maß für die Unsicherheit, hier mit `_` verworfen). Die Standardfehler ergeben sich aus `np.sqrt(np.diag(pcov))`.

In [ ]:
params_1, _ = curve_fit(exp_zerfall, runde_1, verbleibend_1, p0=[99, 0.2])
N0_fit_1, lam_fit_1 = params_1
print(f"Datensatz 1: N0 = {N0_fit_1:.2f}, lambda = {lam_fit_1:.4f} pro Runde")

In [ ]:
t_fein_1 = np.linspace(0, runde_1[-1], 200)

plt.scatter(runde_1, verbleibend_1, label="Messdaten")
plt.plot(t_fein_1, exp_zerfall(t_fein_1, *params_1), color="crimson", label="Fit")
plt.title("Zerfall bei Augenzahl 3 (N0 = 99)")
plt.xlabel("Runde")
plt.ylabel("verbleibende Wuerfel")
plt.legend()
plt.show()

In [ ]:
params_2, _ = curve_fit(exp_zerfall, runde_2, verbleibend_2, p0=[100, 0.2])
N0_fit_2, lam_fit_2 = params_2
print(f"Datensatz 2: N0 = {N0_fit_2:.2f}, lambda = {lam_fit_2:.4f} pro Runde")

In [ ]:
t_fein_2 = np.linspace(0, runde_2[-1], 200)

plt.scatter(runde_2, verbleibend_2, label="Messdaten")
plt.plot(t_fein_2, exp_zerfall(t_fein_2, *params_2), color="crimson", label="Fit")
plt.title("Zerfall bei Augenzahl 6 (N0 = 100)")
plt.xlabel("Runde")
plt.ylabel("verbleibende Wuerfel")
plt.legend()
plt.show()

## Vergleich mit der Theorie

Überlebenswahrscheinlichkeit pro Runde: $5/6$ (5 von 6 Augenzahlen "überleben"). Theoretisch erwartet: $\lambda_{theorie} = -\ln(5/6)$.

In [ ]:
lam_theorie = -np.log(5 / 6)
print(f"lambda_theorie = {lam_theorie:.4f} pro Runde")
print(f"lambda_1       = {lam_fit_1:.4f} pro Runde")
print(f"lambda_2       = {lam_fit_2:.4f} pro Runde")